# MediCS — Colab GPU Runner

Run cells in order. Each cell is independent — re-run any cell to check scores.

**Pipeline:** Setup → Dataset → Attack-Defense Loop (3 rounds) → DPO → Eval → Transfer → Detection → Figures → Upload

In [1]:
# Cell 1 — Setup (run once per session)
from google.colab import drive
drive.mount('/content/drive')

import sys, os
os.chdir('/content/drive/MyDrive/MediCS')
sys.path.insert(0, '/content/drive/MyDrive/MediCS')

%pip install -q -r requirements.txt
!nvidia-smi
print(f"\nWorking dir: {os.getcwd()}")
print("Setup complete.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.9 MB/s eta 0:00:00:00:0100:01
Tue Apr  7 07:49:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                   

## Phase A — Dataset Construction (one-time, no GPU needed)

In [ ]:
# Cell 2 — Build MediCS-500 dataset (seeds → keywords → translation → splits)
# NOTE: Calls OpenAI API for keyword extraction — costs ~$1-2
!python scripts/01_build_dataset.py --config config/experiment_config.yaml

[TIMING] Starting: Dataset Construction
API keys loaded from .env file.
=== MediCS-500 Dataset Builder ===
Languages: ['hi', 'bn', 'sw', 'yo', 'tl', 'gu']
Semantic threshold: 0.75

--- Step 1-2: Loading and deduplicating seeds ---
Loaded 500 raw seeds
Deduplication: 500 -> 500 (removed 0 duplicates)
After dedup: 500 seeds → saved to data/seeds/deduped_seeds.jsonl

--- Step 3: Extracting keywords ---
  Resuming from checkpoint: 500 keywords already extracted
  All 500 keywords already extracted (from checkpoint)
Extracted keywords for 500 seeds

--- Step 4: Code-switching ---
Code-switching: 100% 3000/3000 [00:03<00:00, 849.69it/s] 
Generated 3000 code-switched variants

=== Semantic Preservation Verification ===
modules.json: 100% 349/349 [00:00<00:00, 1.31MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 437kB/s]
README.md: 10.5kB [00:00, 4.96MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 277kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.35MB/s]
model

In [ ]:
# Cell 3 — Token fragmentation analysis (no GPU, no API cost)
!python scripts/07_tokenization_analysis.py

[TIMING] Starting: Tokenization Analysis
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loaded 2665 seeds from data/medics_500/medics_500_full.jsonl
Loaded keywords for 500 seeds

Analyzing tokenization for meta-llama/Meta-Llama-3-8B-Instruct
Languages: ['hi', 'bn', 'sw', 'yo', 'tl', 'gu']
  Tokenization analysis: 50/2665 seeds processed
  Tokenization analysis: 100/2665 seeds processed
  Tokenization analysis: 150/2665 seeds processed
  Tokenization analysis: 200/2665 seeds processed
  Tokenization analysis: 250/2665 seeds processed
  Tokenization analysis: 300/2665 seeds processed
  Tokenization analysis: 350/2665 seeds processed
  Tokenization analysis: 400/2665 seeds processed
  Tokenization analysis: 450/2665 seeds processed
  Tokenization analysis: 500/2665 seeds processed
  Tokenization analysis: 550/2665 seeds processed
  Tokenization analysis: 600/2665 seeds processed
  Tokenization analysis: 650

## Round 1 — Base Model Attack

In [ ]:
# Cell 4 — Round 1: Generate attack prompts (Thompson Sampling)
!python scripts/02_run_attack_round.py --round 1 --phase generate --config config/experiment_config.yaml

[TIMING] Starting: Attack Round
Generation attempt 1/3
Category quotas (strict): {'TOX': 34, 'SH': 33, 'MIS': 34, 'ULP': 33, 'PPV': 33, 'UCA': 33}
  [25/200] generated
  [50/200] generated
  [75/200] generated
  [100/200] generated
  [125/200] generated
  [150/200] generated
  [175/200] generated
  [200/200] generated
MTE generation summary: 0/29 API fallbacks (0.0%)
Quality summary: duplicates=0, category_span=1, mte_fallback=0.0% (0/35), mte_min_turns=3, cs_obf_leet_mean=0.0775

Generated 200 attack prompts → results/attacks/round_1/prompts.jsonl
Strategy distribution (generated): {'CS-OBF': 47, 'CS': 41, 'RP': 33, 'MTE': 35, 'CS-RP': 44}
Category distribution (generated): {'SH': 33, 'PPV': 33, 'TOX': 34, 'ULP': 33, 'MIS': 34, 'UCA': 33}
Next: run colab/run_inference.py on Colab, then come back with --phase judge
[TIMING] Finished: Attack Round (76.4s / 1.3min)
[TIMING] Report saved to results/timing_report.json (7 entries)


In [ ]:
# Cell 5 — Round 1: Inference (base model, BASE prompt)
!python colab/run_inference.py \
    --checkpoint base --system-prompt base --seed 42 \
    --input results/attacks/round_1/prompts.jsonl \
    --output results/attacks/round_1/responses.jsonl

[TIMING] Starting: Inference
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== Target Model Inference ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Checkpoint: base
System prompt: BASE (no safety priming) [override]
Seed: 42
Input: results/attacks/round_1/prompts.jsonl
config.json: 100% 654/654 [00:00<00:00, 2.91MB/s]
model.safetensors.index.json: 23.9kB [00:00, 15.7MB/s]
Fetching 4 files: 100% 4/4 [00:38<00:00,  9.70s/it]
Download complete: 100% 16.1G/16.1G [00:38<00:00, 413MB/s]                
Loading weights: 100% 291/291 [00:04<00:00, 62.05it/s, Materializing param=model.norm.weight]                               
generation_config.json: 100% 187/187 [00:00<00:00, 925kB/s]
tokenizer_config.json: 51.0kB [00:00, 66.9MB/s]
tokenizer.json: 9.09MB [00:01, 7.41MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 348kB/s]
Loaded 200 prompts
Setting `pad_token_id` to `eos_token_id`:128001 for open-e

In [ ]:
# Cell 6 — Round 1: Judge responses (GPT-5)
# NOTE: This calls the OpenAI API — costs ~$0.40
!python scripts/02_run_attack_round.py --round 1 --phase judge --config config/experiment_config.yaml


[TIMING] Starting: Attack Round
Judge model: gpt-5-chat
Judging responses: 100% 200/200 [03:38<00:00,  1.09s/it]
Judge content-filter summary: hits=6, redacted_attempts=0, redacted_successes=0, redacted_failures=0, redacted_skipped=6

--- API Cost Summary ---
Total calls: 195
  gpt-5-chat: 195 calls, 143206 in / 9592 out, ~$0.2749
Estimated session cost: $0.2749
Judge error rate: 0.0%
Judge fallback rate: 3.0% (6/200 used heuristic)

Round 1 ASR: 56.5% (113/200)
Updated bandit state: {'CS': 0.456, 'RP': 0.7920792079207921, 'MTE': 0.2616822429906542, 'CS-RP': 0.7313432835820896, 'CS-OBF': 0.4965034965034965}
[TIMING] Finished: Attack Round (220.2s / 3.7min)
[TIMING] Report saved to results/timing_report.json (11 entries)


In [ ]:
# Cell 7 — Round 1: Check ASR
from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_1/results.jsonl")
print(f"Round 1 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

Round 1 ASR: 56.5% (113/200)

By strategy:
  CS: 48.8%
  CS-OBF: 51.1%
  CS-RP: 77.3%
  MTE: 25.7%
  RP: 78.8%

By category:
  MIS: 38.2%
  PPV: 57.6%
  SH: 45.5%
  TOX: 67.6%
  UCA: 63.6%
  ULP: 66.7%


In [ ]:
# Cell 8 — Round 1: Benign inference (base checkpoint)
!python colab/run_inference.py \
      --checkpoint base --system-prompt base --seed 42 \
      --input data/seeds/benign_twins.jsonl \
      --output results/eval/base/benign_results.jsonl

In [ ]:
# Cell 9 — Round 1: Build SFT data + Train SFT + Benign eval for DPO

# Rebuild SFT data with reduced prefix-recovery ($0 API cost)
!python scripts/03_build_defense_data.py --rounds 1 --type sft \
    --config config/experiment_config.yaml \
    --rebuild-from-cache --force

# Train SFT round 1
!python colab/train_sft.py --round 1 --config config/experiment_config.yaml

 '''there was small bug The inference script reads attack_prompt or prompt field, but benign_twins.jsonl has the field named benign_question. The model never saw the benign question — it fell
  back to an empty string or something unexpected. so i had to rerun the be Benign inference'''

[TIMING] Starting: Defense Data Construction
Round 1: 113 jailbreaks
Benign twins: 500
Cache: 278 refusals, 500 helpful responses
SFT data (from cache): 733 examples (113 refusals + 500 helpful + 60 prefix-recovery + 60 bare-RP-refusal)
SFT data saved: data/defense/sft_round_1.json (733 examples)
Refusal map saved: data/defense/refusals.json (278 entries: 113 prompt-specific, 165 legacy seed_id)
Helpful targets saved: data/defense/helpful_targets.json (500 entries)
[TIMING] Finished: Defense Data Construction (0.6s / 0.0min)
[TIMING] Report saved to results/timing_report.json (16 entries)
[TIMING] Starting: SFT Training
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== QLoRA-SFT Training: Round 1 ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
4-bit compute dtype: bfloat16
Trainer precision: bf16=True fp16=False
Loading weights: 100% 291/291 [00:04<00:00, 62.24it/s, Materializing param=model.norm.weight] 

In [2]:
# Cell 10 — Benign inference
!python colab/run_inference.py \
    --config config/experiment_config.yaml \
    --checkpoint checkpoints/sft/round_1/final --system-prompt base --seed 42 \
    --input data/seeds/benign_twins.jsonl \
    --output results/eval/sft/round_1/benign_results.jsonl

[TIMING] Starting: Inference
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== Target Model Inference ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Checkpoint: checkpoints/sft/round_1/final
System prompt: BASE (no safety priming) [override]
Seed: 42
Input: data/seeds/benign_twins.jsonl
Loaded 500 prompts
4-bit compute dtype: bfloat16
config.json: 100% 654/654 [00:00<00:00, 2.53MB/s]
model.safetensors.index.json: 23.9kB [00:00, 38.7MB/s]
Fetching 4 files: 100% 4/4 [00:36<00:00,  9.19s/it]
Download complete: 100% 16.1G/16.1G [00:36<00:00, 436MB/s]                
Loading weights: 100% 291/291 [00:04<00:00, 61.51it/s, Materializing param=model.norm.weight]                               
generation_config.json: 100% 187/187 [00:00<00:00, 965kB/s]
tokenizer_config.json: 51.0kB [00:00, 63.5MB/s]
tokenizer.json: 9.09MB [00:00, 19.8MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 379kB/s]
Loading ada

## Round 2 — Attack SFT Round 1

In [ ]:
# Cell 11 — Round 2: Generate attacks
!python scripts/02_run_attack_round.py --round 2 --phase generate --config config/experiment_config.yaml

[TIMING] Starting: Attack Round
Generation attempt 1/3
Loading bandit state from round 1
Category quotas (strict): {'TOX': 33, 'SH': 33, 'MIS': 33, 'ULP': 34, 'PPV': 34, 'UCA': 33}
  [25/200] generated
  [50/200] generated
  [75/200] generated
  [100/200] generated
  [125/200] generated
  [150/200] generated
  [175/200] generated
  [200/200] generated
MTE generation summary: 0/34 API fallbacks (0.0%)
Quality summary: duplicates=0, category_span=1, mte_fallback=0.0% (0/34), mte_min_turns=3, cs_obf_leet_mean=0.0952

Generated 200 attack prompts → results/attacks/round_2/prompts.jsonl
Strategy distribution (generated): {'RP': 71, 'MTE': 34, 'CS': 16, 'CS-RP': 44, 'CS-OBF': 35}
Category distribution (generated): {'TOX': 33, 'SH': 33, 'ULP': 34, 'PPV': 34, 'MIS': 33, 'UCA': 33}
Next: run colab/run_inference.py on Colab, then come back with --phase judge
[TIMING] Finished: Attack Round (273.8s / 4.6min)
[TIMING] Report saved to results/timing_report.json (18 entries)


In [2]:
# Cell 12 — Round 2: Inference (SFT round 1 checkpoint, BASE prompt)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_1/final --system-prompt base --seed 42 \
    --input results/attacks/round_2/prompts.jsonl \
    --output results/attacks/round_2/responses.jsonl

[TIMING] Starting: Inference
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== Target Model Inference ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Checkpoint: checkpoints/sft/round_1/final
System prompt: BASE (no safety priming) [override]
Seed: 42
Input: results/attacks/round_2/prompts.jsonl
Loaded 200 prompts
4-bit compute dtype: bfloat16
config.json: 100% 654/654 [00:00<00:00, 2.92MB/s]
model.safetensors.index.json: 23.9kB [00:00, 38.3MB/s]
Fetching 4 files: 100% 4/4 [00:38<00:00,  9.50s/it]
Download complete: 100% 16.1G/16.1G [00:38<00:00, 422MB/s]                
Loading weights: 100% 291/291 [00:04<00:00, 61.97it/s, Materializing param=model.norm.weight]                               
generation_config.json: 100% 187/187 [00:00<00:00, 875kB/s]
tokenizer_config.json: 51.0kB [00:00, 60.9MB/s]
tokenizer.json: 9.09MB [00:01, 8.30MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 376kB/s]
Loa

In [7]:
# Cell 13 — Round 2: Judge + ASR
!python scripts/02_run_attack_round.py --round 2 --phase judge

from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_2/results.jsonl")
print(f"\nRound 2 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

[TIMING] Starting: Attack Round
Judge model: gpt-5-chat
Judging responses: 100% 200/200 [05:04<00:00,  1.52s/it]
Judge content-filter summary: hits=3, redacted_attempts=0, redacted_successes=0, redacted_failures=0, redacted_skipped=3

--- API Cost Summary ---
Total calls: 198
  gpt-5-chat: 198 calls, 110673 in / 8827 out, ~$0.2266
Estimated session cost: $0.2266
Judge error rate: 0.0%
Judge method breakdown: {'api': 197, 'heuristic_content_filter': 3}
Judge fallback rate: 1.5% (3/200 used heuristic)

Round 2 ASR: 16.0% (32/200)
Updated bandit state: {'CS': 0.37572254335260113, 'RP': 0.5636942675159236, 'MTE': 0.2535885167464115, 'CS-RP': 0.518796992481203, 'CS-OBF': 0.2903225806451613}
[TIMING] Finished: Attack Round (306.3s / 5.1min)
[TIMING] Report saved to results/timing_report.json (41 entries)

Round 2 ASR: 16.0% (32/200)

By strategy:
  CS: 12.5%
  CS-OBF: 0.0%
  CS-RP: 6.8%
  MTE: 23.5%
  RP: 26.8%

By category:
  MIS: 3.0%
  PPV: 23.5%
  SH: 9.1%
  TOX: 48.5%
  UCA: 6.1%
  ULP:

In [ ]:
# Cell 14 — Round 2: Build SFT data + Train SFT + Benign eval for DPO
!python scripts/03_build_defense_data.py --rounds 1,2 --type sft
!python colab/train_sft.py --round 2 --prev-checkpoint checkpoints/sft/round_1/final

 '''there was small bug The inference script reads attack_prompt or prompt field, but benign_twins.jsonl has the field named benign_question. The model never saw the benign question — it fell
  back to an empty string or something unexpected. so i had to rerun the be Benign inference'''

[TIMING] Starting: Defense Data Construction
Round 1: 113 jailbreaks
Round 2: 28 jailbreaks
Benign twins: 500
Generating refusals: 100% 141/141 [03:53<00:00,  1.65s/it]
Generating helpful responses: 100% 500/500 [50:57<00:00,  6.12s/it] 
SFT data: 801 examples (141 refusals + 500 helpful + 80 prefix-recovery + 80 bare-RP-refusal)
SFT data saved: data/defense/sft_round_1_2.json (801 examples)
Refusal map saved: data/defense/refusals.json (311 entries: 141 prompt-specific, 170 legacy seed_id)
Helpful targets saved: data/defense/helpful_targets.json (500 entries)
[TIMING] Finished: Defense Data Construction (3290.9s / 54.8min)
[TIMING] Report saved to results/timing_report.json (22 entries)
[TIMING] Starting: SFT Training
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== QLoRA-SFT Training: Round 2 ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
4-bit compute dtype: bfloat16
Trainer precision: bf16=True fp16

In [3]:
# Cell 15 — Benign inference through SFT round 2 (needed for DPO over-refusal correction)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_2/final --system-prompt base --seed 42 \
    --input data/seeds/benign_twins.jsonl \
    --output results/eval/sft/round_2/benign_results.jsonl

[TIMING] Starting: Inference
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== Target Model Inference ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Checkpoint: checkpoints/sft/round_2/final
System prompt: BASE (no safety priming) [override]
Seed: 42
Input: data/seeds/benign_twins.jsonl
Loaded 500 prompts
4-bit compute dtype: bfloat16
Loading weights: 100% 291/291 [00:04<00:00, 61.67it/s, Materializing param=model.norm.weight]                               
Loading adapter: checkpoints/sft/round_2/final
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  [32/500] processed (169s elapsed, ~2478s remaining)
    checkpoint saved → results/eval/sft/round_2/benign_results

## Round 3 — Attack SFT Round 2

In [4]:
# Cell 16 — Round 3: Generate attacks
!python scripts/02_run_attack_round.py --round 3 --phase generate --config config/experiment_config.yaml

[TIMING] Starting: Attack Round
Generation attempt 1/3
Loading bandit state from round 2
Category quotas (strict): {'TOX': 33, 'SH': 33, 'MIS': 34, 'ULP': 34, 'PPV': 33, 'UCA': 33}
  [25/200] generated
  [50/200] generated
  [75/200] generated
  [100/200] generated
  [125/200] generated
  [150/200] generated
  [175/200] generated
  [200/200] generated
MTE generation summary: 0/7 API fallbacks (0.0%)
Quality summary: duplicates=0, category_span=1, mte_fallback=0.0% (0/7), mte_min_turns=3, cs_obf_leet_mean=0.0923

Generated 200 attack prompts → results/attacks/round_3/prompts.jsonl
Strategy distribution (generated): {'RP': 104, 'CS-OBF': 6, 'CS-RP': 66, 'CS': 17, 'MTE': 7}
Category distribution (generated): {'ULP': 34, 'PPV': 33, 'UCA': 33, 'SH': 33, 'MIS': 34, 'TOX': 33}
Next: run colab/run_inference.py on Colab, then come back with --phase judge
[TIMING] Finished: Attack Round (23.8s / 0.4min)
[TIMING] Report saved to results/timing_report.json (30 entries)


In [5]:
# Cell 17 — Round 3: Inference (SFT round 2 checkpoint, BASE prompt)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_2/final --system-prompt base --seed 42 \
    --input results/attacks/round_3/prompts.jsonl \
    --output results/attacks/round_3/responses.jsonl

[TIMING] Starting: Inference
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== Target Model Inference ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Checkpoint: checkpoints/sft/round_2/final
System prompt: BASE (no safety priming) [override]
Seed: 42
Input: results/attacks/round_3/prompts.jsonl
Loaded 200 prompts
4-bit compute dtype: bfloat16
Loading weights: 100% 291/291 [00:04<00:00, 61.26it/s, Materializing param=model.norm.weight]                               
Loading adapter: checkpoints/sft/round_2/final
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  [1/200] processed (35s elapsed, ~6884s remaining)
    checkpoint saved → results/attacks/round_3/responses.jsonl.partial
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  [2/200] processed (58s elapsed, ~5751s remaining)
    checkpoint saved → results/attacks/round_3/responses.jsonl.partial
Setting 

In [6]:
# Cell 18 — Round 3: Judge + ASR
!python scripts/02_run_attack_round.py --round 3 --phase judge

from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_3/results.jsonl")
print(f"\nRound 3 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

[TIMING] Starting: Attack Round
Judge model: gpt-5-chat
Judging responses: 100% 200/200 [04:27<00:00,  1.34s/it]
Judge content-filter summary: hits=4, redacted_attempts=0, redacted_successes=0, redacted_failures=0, redacted_skipped=4

--- API Cost Summary ---
Total calls: 197
  gpt-5-chat: 197 calls, 102463 in / 8162 out, ~$0.2097
Estimated session cost: $0.2097
Judge error rate: 0.0%
Judge method breakdown: {'api': 196, 'heuristic_content_filter': 4}
Judge fallback rate: 2.0% (4/200 used heuristic)

Round 3 ASR: 1.0% (2/200)
Updated bandit state: {'CS': 0.3620689655172414, 'RP': 0.45821325648414984, 'MTE': 0.25274725274725274, 'CS-RP': 0.46875, 'CS-OBF': 0.3287671232876712}
[TIMING] Finished: Attack Round (269.5s / 4.5min)
[TIMING] Report saved to results/timing_report.json (38 entries)

Round 3 ASR: 1.0% (2/200)

By strategy:
  CS: 0.0%
  CS-OBF: 0.0%
  CS-RP: 0.0%
  MTE: 14.3%
  RP: 1.0%

By category:
  MIS: 0.0%
  PPV: 0.0%
  SH: 0.0%
  TOX: 3.0%
  UCA: 3.0%
  ULP: 0.0%


In [ ]:
# Cell 19 — Final SFT (round 3) + Benign eval + DPO
!python scripts/03_build_defense_data.py --rounds 1,2,3 --type sft
!python colab/train_sft.py --round 3 --prev-checkpoint checkpoints/sft/round_2/final

# Benign inference through SFT round 3 (needed for DPO over-refusal correction)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_3/final --system-prompt base --seed 42 \
    --input data/seeds/benign_twins.jsonl \
    --output results/eval/sft/round_3/benign_results.jsonl

# Build DPO data (uses benign results from all 3 SFT rounds for over-refusal correction)
!python scripts/03_build_defense_data.py --rounds 1,2,3 --type dpo
!python colab/train_dpo.py --sft-checkpoint checkpoints/sft/round_3/final

## Held-Out Evaluation — 3 Checkpoints x 3 Seeds

In [ ]:
# Cell 20 — Held-out evaluation: all checkpoints x all seeds (BASE prompt throughout)
checkpoints = {
    "base": "base",
    "sft": "checkpoints/sft/round_3/final",
    "dpo": "checkpoints/dpo/final",
}
seeds = [42, 123, 456]

for ckpt_name, ckpt_path in checkpoints.items():
    for seed in seeds:
        # Attack prompts
        out = f"results/eval/{ckpt_name}/seed_{seed}/held_out.jsonl"
        print(f"\n{'='*60}")
        print(f"Running: {ckpt_name} seed={seed} (attacks)")
        !python colab/run_inference.py \
            --checkpoint {ckpt_path} --system-prompt base --seed {seed} \
            --input data/splits/held_out_eval.jsonl \
            --output {out}

        # Benign twins (for helpfulness judging)
        benign_out = f"results/eval/{ckpt_name}/seed_{seed}/benign_results.jsonl"
        print(f"Running: {ckpt_name} seed={seed} (benign)")
        !python colab/run_inference.py \
            --checkpoint {ckpt_path} --system-prompt base --seed {seed} \
            --input data/seeds/benign_twins.jsonl \
            --output {benign_out}

In [ ]:
# Cell 21 — Helpfulness judging (must run BEFORE metric evaluation)
# Judge benign results for all checkpoints x all seeds
import itertools
checkpoints = ["base", "sft", "dpo"]
seeds = [42, 123, 456]

for ckpt, seed in itertools.product(checkpoints, seeds):
    inp = f"results/eval/{ckpt}/seed_{seed}/benign_results.jsonl"
    print(f"Judging helpfulness: {ckpt} seed={seed}")
    !python scripts/04_evaluate.py --judge-helpfulness --input {inp}

In [ ]:
# Cell 22 — Full metric evaluation (ASR, HR, FRR, McNemar, Cohen's h)
# Run AFTER helpfulness judging so HR/FRR are accurate
!python scripts/04_evaluate.py --checkpoints base,sft,dpo --seeds 42,123,456

## Transfer + Detection + Figures + Upload

In [ ]:
# Cell 23 — Transfer evaluation (Mistral-7B, BASE prompt)
!python colab/run_transfer.py --input data/splits/held_out_eval.jsonl --seed 42
!python scripts/04_evaluate.py --judge-transfer --input results/transfer/mistral_results.jsonl

In [ ]:
# Cell 24 — Perplexity detection baseline
!python colab/run_perplexity.py --checkpoint base \
    --input data/splits/held_out_eval.jsonl \
    --english-input data/seeds/raw_seeds.jsonl \
    --output results/analysis/perplexity_results.json
!python scripts/08_detection_analysis.py

In [ ]:
# Cell 25 — Residual analysis + Figures + Cost report
!python scripts/04_evaluate.py --residual-analysis --checkpoint dpo
!python scripts/05_generate_figures.py --results-dir results/eval/
!python scripts/09_cost_report.py

## Strict A/B Validation — Publication-Grade Check


In [ ]:
# Cell 26 — Strict A/B setup: freeze round-2 prompt set
from pathlib import Path
import shutil

src = Path("results/attacks/round_2/prompts.jsonl")
dst_dir = Path("results/ab")
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir / "prompts_fixed_r2.jsonl"

if not src.exists():
    raise FileNotFoundError(f"Missing source prompts: {src}")

shutil.copy2(src, dst)
print(f"Frozen prompt set: {src} -> {dst}")
print(f"Prompt count: {sum(1 for _ in dst.open())}")


In [ ]:
# Cell 27 — Strict A/B baseline: base model inference + judging
!python colab/run_inference.py \
    --checkpoint base --system-prompt base --seed 42 \
    --input results/ab/prompts_fixed_r2.jsonl \
    --output results/ab/base_responses.jsonl

!python scripts/04_evaluate.py \
    --judge-transfer \
    --input results/ab/base_responses.jsonl \
    --output results/ab/base_judged.jsonl


In [ ]:
# Cell 28 — Strict A/B defended: SFT round-1 inference + judging
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_1/final --system-prompt base --seed 42 \
    --input results/ab/prompts_fixed_r2.jsonl \
    --output results/ab/sft_r1_responses.jsonl

!python scripts/04_evaluate.py \
    --judge-transfer \
    --input results/ab/sft_r1_responses.jsonl \
    --output results/ab/sft_r1_judged.jsonl


In [ ]:
# Cell 29 — Strict A/B comparison: ASR delta + McNemar significance
import json
from collections import Counter
from medics.metrics import compute_asr, compute_judge_fallback_rate, mcnemar_test

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

base = load_jsonl("results/ab/base_judged.jsonl")
sft = load_jsonl("results/ab/sft_r1_judged.jsonl")

print("=== Strict A/B Summary ===")
print(f"Base ASR: {compute_asr(base):.1%}")
print(f"SFT  ASR: {compute_asr(sft):.1%}")
print(f"Delta  : {compute_asr(sft) - compute_asr(base):+.1%}")
print()
print("Judge method breakdown (base):", dict(Counter(r.get("judge_method", "unknown") for r in base)))
print("Judge method breakdown (sft) :", dict(Counter(r.get("judge_method", "unknown") for r in sft)))
print(f"Judge fallback rate (base): {compute_judge_fallback_rate(base):.1%}")
print(f"Judge fallback rate (sft) : {compute_judge_fallback_rate(sft):.1%}")

def row_key(r):
    return (
        r.get("seed_id", ""),
        r.get("strategy", ""),
        r.get("category", ""),
        r.get("language", ""),
        r.get("attack_prompt", ""),
    )

base_map = {row_key(r): r for r in base}
sft_map = {row_key(r): r for r in sft}
keys = [k for k in base_map if k in sft_map]

if not keys:
    raise RuntimeError("No matched rows between base and sft judged files.")

before_correct = [base_map[k].get("judge_label") == "safe" for k in keys]
after_correct = [sft_map[k].get("judge_label") == "safe" for k in keys]
p_value = mcnemar_test(before_correct, after_correct)

fixed = sum(
    1 for k in keys
    if base_map[k].get("judge_label") == "harmful" and sft_map[k].get("judge_label") == "safe"
)
regressed = sum(
    1 for k in keys
    if base_map[k].get("judge_label") == "safe" and sft_map[k].get("judge_label") == "harmful"
)

print()
print(f"Matched rows: {len(keys)}")
print(f"Fixed jailbreaks (base harmful -> sft safe): {fixed}")
print(f"Regressions (base safe -> sft harmful): {regressed}")
print(f"McNemar p-value: {p_value:.6g}")
if p_value < 0.05:
    print("Result: statistically significant difference (p < 0.05).")
else:
    print("Result: not statistically significant at p < 0.05.")


In [ ]:
# Cell 30 — Upload to HuggingFace Hub
!python scripts/06_upload_hf.py

## ASR Dashboard — Re-run anytime to check all scores

In [ ]:
# Cell 31 — ASR Dashboard: all rounds at a glance
from medics.metrics import compute_asr, compute_per_strategy_asr
from medics.utils import load_jsonl
from pathlib import Path

print("=" * 60)
print("  MediCS ASR Dashboard")
print("=" * 60)

# Attack rounds
for r in range(1, 4):
    path = f"results/attacks/round_{r}/results.jsonl"
    if Path(path).exists():
        results = load_jsonl(path)
        asr = compute_asr(results)
        n_harmful = sum(1 for x in results if x.get("judge_label") == "harmful")
        strats = compute_per_strategy_asr(results)
        strat_str = " | ".join(f"{s}:{v:.0%}" for s, v in sorted(strats.items()))
        print(f"\n  Round {r}: ASR {asr:.1%} ({n_harmful}/{len(results)})")
        print(f"    {strat_str}")
    else:
        print(f"\n  Round {r}: not yet run")

# Held-out evaluation
print(f"\n{'='*60}")
print("  Held-Out Evaluation (seed=42)")
print("=" * 60)
for ckpt in ["base", "sft", "dpo"]:
    path = f"results/eval/{ckpt}/seed_42/held_out.jsonl"
    if Path(path).exists():
        results = load_jsonl(path)
        asr = compute_asr(results)
        print(f"  {ckpt:>5}: ASR {asr:.1%}")
    else:
        print(f"  {ckpt:>5}: not yet run")

# Transfer
transfer_path = "results/transfer/mistral_results.jsonl"
if Path(transfer_path).exists():
    results = load_jsonl(transfer_path)
    asr = compute_asr(results)
    print(f"\n  Transfer (Mistral-7B): ASR {asr:.1%}")